---
title: "Project 02 · The Embedding Laboratory"
subtitle: "Inspect, train, and compare word representations"
author: "CS 351 · Starter notebook"
format:
  html:
    toc: true
    toc-depth: 2
jupyter: python3
execute:
  enabled: false
---

## Before you begin

This project moves through three levels of difficulty:

1. **Explore** a mature pretrained embedding.
2. **Train** Skip-gram Word2Vec step by step with PyTorch.
3. **Adapt** the pipeline to GloVe by completing three carefully guided pieces.

The infrastructure, plotting, batching, evaluation, and training loops are supplied. Your work is concentrated in the functions marked **TODO**. Every TODO is followed by a small check; make that check pass before continuing.

The first run downloads GloVe and `text8`. Later runs reuse the local Gensim cache. On Colab, select **Runtime → Run all** after completing the TODOs. A GPU is optional: the defaults are intentionally manageable on CPU.

### Learning goals

By the end, you should be able to:

- query and visualize a pretrained embedding space;
- turn a token sequence into positive and negative Skip-gram pairs;
- explain why two embedding tables are trained;
- train a small embedding model with binary cross-entropy;
- build a weighted co-occurrence dataset; and
- contrast Word2Vec's local prediction objective with GloVe's global count objective.

## Setup

In [ ]:
#| label: imports
from __future__ import annotations

import math
import random
from collections import Counter, defaultdict
from dataclasses import dataclass
from typing import Iterable, Sequence

import gensim.downloader as api
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.decomposition import PCA
from torch import nn
from torch.utils.data import DataLoader, Dataset, TensorDataset

SEED = 351
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} · device: {DEVICE}")

::: {.callout-note}
If imports fail on Colab, run this once, restart the runtime, and run from the top:

```python
%pip install -q gensim matplotlib pandas scikit-learn "smart-open<8" torch
```
:::

# Part I · Explore a known embedding

We begin with `glove-wiki-gigaword-50`: 50-dimensional GloVe vectors trained from Wikipedia and Gigaword. Gensim supplies the download and a `KeyedVectors` interface. You will complete exactly three tasks; the similarity calculations, plotting functions, and interpretation scaffolds are supplied.

## TODO 1 · Load the pretrained vectors

Complete `load_pretrained_vectors()` so that it returns the named Gensim model.

In [ ]:
def load_pretrained_vectors(model_name: str = "glove-wiki-gigaword-50"):
    """Download (once), cache, and return a Gensim KeyedVectors model."""
    # TODO 1: replace the line below.
    raise NotImplementedError("Return the model loaded by gensim.downloader")


glove = load_pretrained_vectors()
print(f"Vocabulary: {len(glove):,} words")
print(f"Dimension:  {glove.vector_size}")

<details>
<summary><strong>Hint 1 — which function?</strong></summary>

The module was imported as `api`. Its `load(...)` function accepts a model name and returns the downloaded object:

```python
model = api.load(model_name)
```

Your function needs only one return statement. The first execution downloads the vectors; this is expected.
</details>

In [ ]:
assert glove.vector_size == 50
assert "king" in glove and "queen" in glove
print("✓ TODO 1 passed")

The next cell gives you a reusable neighborhood table. You do not need to modify it.

In [ ]:
def neighbor_table(vectors, queries: Sequence[str], topn: int = 5) -> pd.DataFrame:
    rows = []
    for query in queries:
        for rank, (neighbor, similarity) in enumerate(
            vectors.most_similar(query, topn=topn), start=1
        ):
            rows.append(
                {
                    "query": query,
                    "rank": rank,
                    "neighbor": neighbor,
                    "cosine_similarity": similarity,
                }
            )
    return pd.DataFrame(rows)


neighbor_table(glove, ["coffee", "language", "doctor"])

## TODO 2 · Perform vector arithmetic

Complete the analogy function. It should find words close to

$$
\mathbf{a}-\mathbf{b}+\mathbf{c}.
$$

For `king − man + woman`, pass the words that should be **added** in `positive` and the word that should be **subtracted** in `negative`.

In [ ]:
def solve_analogy(vectors, a: str, b: str, c: str, topn: int = 5):
    """Return candidates for a - b + c."""
    # TODO 2: replace the line below.
    raise NotImplementedError("Use vectors.most_similar with positive and negative")


royalty_analogy = solve_analogy(glove, "king", "man", "woman")
royalty_analogy

<details>
<summary><strong>Hint 2 — translate the signs literally</strong></summary>

Gensim expects two lists:

- words with a plus sign go in `positive=[...]`;
- words with a minus sign go in `negative=[...]`.

Therefore, `a - b + c` becomes:

```python
vectors.most_similar(positive=[a, c], negative=[b], topn=topn)
```
</details>

In [ ]:
assert len(royalty_analogy) == 5
assert royalty_analogy[0][0] == "queen"
assert all(isinstance(score, float) for _, score in royalty_analogy)
print("✓ TODO 2 passed")

Try two additional relationships. Treat the results as corpus evidence—not as logical proof.

In [ ]:
print("Paris - France + Italy:", solve_analogy(glove, "paris", "france", "italy")[:3])
print("Walking - walk + swim:", solve_analogy(glove, "walking", "walk", "swim")[:3])

## TODO 3 · Project selected vectors into two dimensions

Complete `project_words()`. It should:

1. retrieve the original vectors in the same order as `words`;
2. create a two-component PCA model using `SEED`;
3. fit PCA and return a NumPy array with shape `(len(words), 2)`.

The plotting code is already written.

In [ ]:
WORDS_TO_PLOT = [
    "king", "queen", "prince", "princess",
    "coffee", "tea", "juice", "water",
    "car", "bus", "train", "bicycle",
]


def project_words(vectors, words: Sequence[str]) -> np.ndarray:
    """Return a two-dimensional PCA projection for words, preserving order."""
    # TODO 3: replace the line below.
    raise NotImplementedError("Retrieve vectors, create PCA(2), and fit_transform")


pretrained_points = project_words(glove, WORDS_TO_PLOT)

<details>
<summary><strong>Hint 3 — three short lines</strong></summary>

`KeyedVectors` supports list indexing, so `vectors[list(words)]` returns a matrix with one row per word. Then use:

```python
pca = PCA(n_components=2, random_state=SEED)
points = pca.fit_transform(matrix)
return points
```

PCA is fitted only for visualization. Similarity must still be evaluated in the original 50-dimensional space.
</details>

In [ ]:
assert pretrained_points.shape == (len(WORDS_TO_PLOT), 2)
assert np.isfinite(pretrained_points).all()
print("✓ TODO 3 passed")

In [ ]:
def plot_embedding(
    words: Sequence[str],
    points: np.ndarray,
    title: str,
    groups: Sequence[str] | None = None,
):
    fig, ax = plt.subplots(figsize=(10, 7))
    if groups is None:
        groups = ["words"] * len(words)

    group_names = list(dict.fromkeys(groups))
    palette = plt.cm.Set2(np.linspace(0, 1, len(group_names)))
    color_for = dict(zip(group_names, palette))

    for word, (x, y), group in zip(words, points, groups):
        ax.scatter(x, y, s=75, color=color_for[group], label=group)
        ax.annotate(word, (x, y), xytext=(5, 5), textcoords="offset points")

    handles, labels = ax.get_legend_handles_labels()
    unique = dict(zip(labels, handles))
    ax.legend(unique.values(), unique.keys(), frameon=False)
    ax.axhline(0, color="#d9e3e1", linewidth=0.8)
    ax.axvline(0, color="#d9e3e1", linewidth=0.8)
    ax.set_title(title)
    ax.set_xlabel("Principal component 1")
    ax.set_ylabel("Principal component 2")
    plt.tight_layout()
    plt.show()


plot_embedding(
    WORDS_TO_PLOT,
    pretrained_points,
    "Pretrained GloVe vectors projected with PCA",
    groups=["royalty"] * 4 + ["drinks"] * 4 + ["transport"] * 4,
)

### Part I interpretation

Answer briefly:

1. Are the three semantic groups visibly separated? Identify one exception.
2. Why should you not estimate cosine similarity from this PCA plot?
3. Find one nearest-neighbor result that reflects topical association rather than synonymy.

# Part II · Train Word2Vec step by step

We now learn vectors rather than download them. We use the first 40,000 tokens of `text8`, a cleaned English Wikipedia corpus. This is deliberately a **small teaching slice**. The resulting vectors will be noisier than published embeddings, but the complete training cycle remains manageable.

## Load and prepare the corpus

In [ ]:
MAX_TOKENS = 40_000
MAX_VOCAB = 2_500
MIN_COUNT = 3
WINDOW_SIZE = 2
NEGATIVES_PER_POSITIVE = 3

text8 = api.load("text8")
tokens = []
for document in text8:
    remaining = MAX_TOKENS - len(tokens)
    if remaining <= 0:
        break
    tokens.extend(document[:remaining])

raw_counts = Counter(tokens)
vocabulary = [
    word
    for word, count in raw_counts.most_common(MAX_VOCAB)
    if count >= MIN_COUNT
]
word_to_id = {word: index for index, word in enumerate(vocabulary)}
id_to_word = dict(enumerate(vocabulary))
token_ids = [word_to_id[word] for word in tokens if word in word_to_id]

print(f"Raw tokens:       {len(tokens):,}")
print(f"Retained tokens:  {len(token_ids):,}")
print(f"Vocabulary size:  {len(vocabulary):,}")
print("Most common:", raw_counts.most_common(10))

## TODO 4 · Create positive Skip-gram pairs

For every center position, collect context positions within `window_size`, excluding the center itself and positions outside the sequence.

In [ ]:
def make_skipgram_pairs(
    ids: Sequence[int], window_size: int
) -> tuple[torch.Tensor, torch.Tensor]:
    """Return aligned tensors of center IDs and observed context IDs."""
    centers: list[int] = []
    contexts: list[int] = []

    # TODO 4: complete the nested loop.
    # The return statement is supplied.

    return torch.tensor(centers), torch.tensor(contexts)


positive_centers, positive_contexts = make_skipgram_pairs(token_ids, WINDOW_SIZE)
print(f"Positive pairs: {len(positive_centers):,}")

<details>
<summary><strong>Hint 4A — determine valid boundaries</strong></summary>

At center position `i`, valid context indices start at
`max(0, i - window_size)` and stop before
`min(len(ids), i + window_size + 1)`.
</details>

<details>
<summary><strong>Hint 4B — near-complete loop</strong></summary>

```python
for i, center_id in enumerate(ids):
    left = max(0, i - window_size)
    right = min(len(ids), i + window_size + 1)
    for j in range(left, right):
        if ...:                  # exclude the center position
            continue
        centers.append(...)
        contexts.append(...)
```

Append `center_id` and `ids[j]` in the final two lines.
</details>

In [ ]:
toy_centers, toy_contexts = make_skipgram_pairs([10, 20, 30], window_size=1)
assert list(zip(toy_centers.tolist(), toy_contexts.tolist())) == [
    (10, 20), (20, 10), (20, 30), (30, 20)
]
assert len(positive_centers) == len(positive_contexts) > 0
print("✓ TODO 4 passed")

## TODO 5 · Sample negative contexts

The supplied distribution uses frequency raised to the $3/4$ power. Complete the function by sampling `count` vocabulary IDs **with replacement**.

In [ ]:
vocab_counts = torch.tensor(
    [raw_counts[word] for word in vocabulary], dtype=torch.float
)
noise_distribution = vocab_counts.pow(0.75)
noise_distribution /= noise_distribution.sum()


def sample_negative_contexts(count: int) -> torch.Tensor:
    """Draw negative context IDs from the smoothed unigram distribution."""
    # TODO 5: replace the line below.
    raise NotImplementedError("Use torch.multinomial with replacement=True")

<details>
<summary><strong>Hint 5 — PyTorch already implements sampling</strong></summary>

Use `torch.multinomial(distribution, num_samples, replacement=True)`:

```python
torch.multinomial(noise_distribution, count, replacement=True)
```

Why replacement? The same common context may be sampled more than once across independent noise examples.
</details>

In [ ]:
sampled = sample_negative_contexts(100)
assert sampled.shape == (100,)
assert sampled.dtype == torch.int64
assert sampled.min() >= 0 and sampled.max() < len(vocabulary)
print("✓ TODO 5 passed")

## Assemble a balanced training interface

The next function is supplied. It combines every positive pair with three sampled negative pairs and labels them `1` or `0`. To keep memory modest, it creates negatives once for this teaching run.

In [ ]:
def build_word2vec_dataset(
    centers: torch.Tensor,
    contexts: torch.Tensor,
    negatives_per_positive: int,
) -> TensorDataset:
    negative_centers = centers.repeat_interleave(negatives_per_positive)
    negative_contexts = sample_negative_contexts(len(negative_centers))

    all_centers = torch.cat([centers, negative_centers])
    all_contexts = torch.cat([contexts, negative_contexts])
    labels = torch.cat(
        [torch.ones(len(centers)), torch.zeros(len(negative_centers))]
    )

    order = torch.randperm(len(labels))
    return TensorDataset(all_centers[order], all_contexts[order], labels[order])


word2vec_dataset = build_word2vec_dataset(
    positive_centers, positive_contexts, NEGATIVES_PER_POSITIVE
)
word2vec_loader = DataLoader(word2vec_dataset, batch_size=2_048, shuffle=True)

batch_centers, batch_contexts, batch_labels = next(iter(word2vec_loader))
print(batch_centers.shape, batch_contexts.shape, batch_labels.shape)
print("Positive fraction:", batch_labels.mean().item())

## TODO 6 · Implement the Skip-gram score

The model has two embedding tables. In `forward`, look up both batches and return one dot product per pair.

In [ ]:
class SkipGramNegativeSampling(nn.Module):
    def __init__(self, vocabulary_size: int, embedding_dim: int):
        super().__init__()
        self.center_embeddings = nn.Embedding(vocabulary_size, embedding_dim)
        self.context_embeddings = nn.Embedding(vocabulary_size, embedding_dim)
        nn.init.normal_(self.center_embeddings.weight, std=0.02)
        nn.init.normal_(self.context_embeddings.weight, std=0.02)

    def forward(
        self, center_ids: torch.Tensor, context_ids: torch.Tensor
    ) -> torch.Tensor:
        # TODO 6: replace the line below.
        raise NotImplementedError("Look up both vectors and compute row-wise dots")

<details>
<summary><strong>Hint 6A — inspect the shapes</strong></summary>

For a batch of `B` pairs and embedding dimension `D`:

- `self.center_embeddings(center_ids)` has shape `(B, D)`;
- `self.context_embeddings(context_ids)` has shape `(B, D)`;
- multiplying them element by element keeps shape `(B, D)`;
- summing over `dim=1` produces the required `(B,)` scores.
</details>

<details>
<summary><strong>Hint 6B — almost the complete method</strong></summary>

```python
center_vectors = self.center_embeddings(center_ids)
context_vectors = self.context_embeddings(context_ids)
return (center_vectors * context_vectors).sum(dim=...)
```

Choose the dimension containing embedding coordinates, not the batch dimension.
</details>

In [ ]:
test_model = SkipGramNegativeSampling(20, 8)
test_scores = test_model(torch.tensor([1, 2, 3]), torch.tensor([4, 5, 6]))
assert test_scores.shape == (3,)
assert test_scores.requires_grad
print("✓ TODO 6 passed")

## TODO 7 · Complete one optimization step

Use the supplied loss function to compare raw model scores with binary labels. Then perform the standard PyTorch gradient steps in the correct order.

In [ ]:
def train_word2vec_epoch(model, loader, optimizer, loss_fn) -> float:
    model.train()
    total_loss = 0.0

    for center_ids, context_ids, labels in loader:
        center_ids = center_ids.to(DEVICE)
        context_ids = context_ids.to(DEVICE)
        labels = labels.to(DEVICE)

        # TODO 7: write five lines:
        # 1. clear old gradients
        # 2. compute scores
        # 3. compute loss
        # 4. backpropagate
        # 5. update parameters

        total_loss += loss.item() * len(labels)

    return total_loss / len(loader.dataset)

<details>
<summary><strong>Hint 7 — the standard PyTorch sequence</strong></summary>

Fill the blank with these operations, replacing argument names if necessary:

```python
optimizer.zero_grad()
scores = model(center_ids, context_ids)
loss = loss_fn(scores, labels)
loss.backward()
optimizer.step()
```

`BCEWithLogitsLoss` expects raw dot-product scores. Do **not** apply sigmoid yourself.
</details>

In [ ]:
EMBEDDING_DIM = 50
WORD2VEC_EPOCHS = 4

word2vec = SkipGramNegativeSampling(len(vocabulary), EMBEDDING_DIM).to(DEVICE)
word2vec_optimizer = torch.optim.Adam(word2vec.parameters(), lr=0.01)
binary_loss = nn.BCEWithLogitsLoss()

word2vec_history = []
for epoch in range(1, WORD2VEC_EPOCHS + 1):
    mean_loss = train_word2vec_epoch(
        word2vec, word2vec_loader, word2vec_optimizer, binary_loss
    )
    word2vec_history.append(mean_loss)
    print(f"Epoch {epoch:02d} · loss {mean_loss:.4f}")

In [ ]:
assert len(word2vec_history) == WORD2VEC_EPOCHS
assert np.isfinite(word2vec_history).all()
assert word2vec_history[-1] < word2vec_history[0]
print("✓ TODO 7 passed")

plt.plot(range(1, WORD2VEC_EPOCHS + 1), word2vec_history, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Mean binary loss")
plt.title("Word2Vec training curve")
plt.show()

## Inspect the learned weights

The rows of `center_embeddings.weight` are the learned vectors. The helper functions below are supplied and will also be reused for GloVe.

In [ ]:
def normalized_rows(matrix: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    return matrix / np.clip(norms, 1e-12, None)


def nearest_words(
    matrix: np.ndarray,
    query: str,
    word_to_id: dict[str, int],
    id_to_word: dict[int, str],
    topn: int = 5,
):
    if query not in word_to_id:
        return []
    normalized = normalized_rows(matrix)
    query_id = word_to_id[query]
    scores = normalized @ normalized[query_id]
    best = np.argsort(-scores)
    return [
        (id_to_word[index], float(scores[index]))
        for index in best
        if index != query_id
    ][:topn]


word2vec_weights = word2vec.center_embeddings.weight.detach().cpu().numpy()
print("Weight matrix:", word2vec_weights.shape)

for query in ["king", "queen", "water", "music"]:
    print(query, nearest_words(word2vec_weights, query, word_to_id, id_to_word))

## Visualize the learned Word2Vec space

The vocabulary of a small corpus may not contain every preferred word. This cell chooses available frequent words automatically and uses the plotting function from Part I.

In [ ]:
candidate_words = [
    "king", "queen", "man", "woman", "water", "food",
    "music", "science", "city", "country", "war", "peace",
]
learned_words = [word for word in candidate_words if word in word_to_id]

learned_matrix = np.stack(
    [word2vec_weights[word_to_id[word]] for word in learned_words]
)
learned_points = PCA(n_components=2, random_state=SEED).fit_transform(learned_matrix)
plot_embedding(learned_words, learned_points, "Word2Vec weights learned from 40k tokens")

### Part II interpretation

1. Identify one neighborhood that appears semantically or syntactically coherent.
2. Identify one poor neighbor. How might corpus size explain it?
3. Why do these vectors not match the pretrained GloVe quality from Part I?
4. Explain in plain language why negative sampling makes training cheaper.

# Part III · Build GloVe from global counts

Word2Vec repeatedly observes local pairs. GloVe begins with their **aggregated co-occurrence counts**. Most of the pipeline is supplied. You complete exactly three places:

1. update weighted co-occurrence counts;
2. implement the GloVe weighting function;
3. compute predictions and the weighted squared loss.

## GloVe TODO 1 of 3 · Update co-occurrence counts

For every center and valid context position, add inverse-distance weight

$$
\frac{1}{|i-j|}
$$

to `counts[(center_id, context_id)]`.

In [ ]:
def build_cooccurrence(
    ids: Sequence[int], window_size: int
) -> dict[tuple[int, int], float]:
    counts: defaultdict[tuple[int, int], float] = defaultdict(float)

    for i, center_id in enumerate(ids):
        left = max(0, i - window_size)
        right = min(len(ids), i + window_size + 1)

        for j in range(left, right):
            if i == j:
                continue

            context_id = ids[j]
            distance = abs(i - j)

            # GLOVE TODO 1: add inverse-distance weight to this pair.
            # Replace the next line.
            raise NotImplementedError("Update counts[(center_id, context_id)]")

    return dict(counts)

<details>
<summary><strong>GloVe Hint 1A — identify the dictionary key</strong></summary>

The row/column identity is the tuple `(center_id, context_id)`. Because `counts` is a `defaultdict(float)`, an unseen key starts at `0.0` automatically.
</details>

<details>
<summary><strong>GloVe Hint 1B — the missing statement</strong></summary>

Use an in-place addition:

```python
counts[(center_id, context_id)] += 1.0 / distance
```

Immediate neighbors contribute `1`; words two positions away contribute `0.5`. This makes closer context stronger.
</details>

In [ ]:
toy_counts = build_cooccurrence([0, 1, 2], window_size=2)
assert math.isclose(toy_counts[(0, 1)], 1.0)
assert math.isclose(toy_counts[(0, 2)], 0.5)
assert (1, 1) not in toy_counts
print("✓ GloVe TODO 1 passed")

cooccurrence = build_cooccurrence(token_ids, window_size=WINDOW_SIZE)
print(f"Nonzero co-occurrence pairs: {len(cooccurrence):,}")

The supplied dataset converts the dictionary into aligned tensors.

In [ ]:
def cooccurrence_dataset(
    counts: dict[tuple[int, int], float]
) -> TensorDataset:
    pairs = list(counts)
    center_ids = torch.tensor([pair[0] for pair in pairs])
    context_ids = torch.tensor([pair[1] for pair in pairs])
    values = torch.tensor([counts[pair] for pair in pairs], dtype=torch.float)
    return TensorDataset(center_ids, context_ids, values)


glove_dataset = cooccurrence_dataset(cooccurrence)
glove_loader = DataLoader(glove_dataset, batch_size=2_048, shuffle=True)
print(f"GloVe training rows: {len(glove_dataset):,}")

## GloVe TODO 2 of 3 · Weight each count

GloVe uses

$$
f(x)=
\begin{cases}
(x/x_{max})^\alpha & x < x_{max},\\
1 & x \ge x_{max}.
\end{cases}
$$

Implement this element by element with PyTorch. The result must have the same shape as `counts`.

In [ ]:
def glove_weight(
    counts: torch.Tensor,
    x_max: float = 10.0,
    alpha: float = 0.75,
) -> torch.Tensor:
    """Downweight rare counts and cap weights at one."""
    # GLOVE TODO 2: replace the line below.
    raise NotImplementedError("Use torch.clamp or torch.where")

<details>
<summary><strong>GloVe Hint 2A — reason from boundary cases</strong></summary>

- If `count == x_max`, the weight must be `1`.
- If `count > x_max`, it must remain `1`.
- If `count == 1` and `x_max == 10`, it should be `0.1 ** alpha`.

Compute `(counts / x_max).pow(alpha)` and cap values at `1.0`.
</details>

<details>
<summary><strong>GloVe Hint 2B — one-line solution shape</strong></summary>

Either form works:

```python
torch.clamp((counts / x_max).pow(alpha), max=1.0)
```

or a `torch.where(counts < x_max, ..., 1)` expression.
</details>

In [ ]:
test_counts = torch.tensor([1.0, 10.0, 100.0])
test_weights = glove_weight(test_counts, x_max=10.0, alpha=0.75)
assert test_weights.shape == test_counts.shape
assert 0 < test_weights[0] < 1
assert torch.allclose(test_weights[1:], torch.ones(2))
print("✓ GloVe TODO 2 passed")

## GloVe TODO 3 of 3 · Predict log counts and compute loss

The model parameters are supplied:

- a center embedding $\mathbf{w}_i$;
- a context embedding $\widetilde{\mathbf{w}}_j$;
- one bias for every center word;
- one bias for every context word.

Complete `forward` using

$$
\widehat{\log X_{ij}}
=\mathbf{w}_i^T\widetilde{\mathbf{w}}_j+b_i+\widetilde b_j,
$$

then return the mean weighted squared error.

In [ ]:
class GloVeModel(nn.Module):
    def __init__(self, vocabulary_size: int, embedding_dim: int):
        super().__init__()
        self.word_embeddings = nn.Embedding(vocabulary_size, embedding_dim)
        self.context_embeddings = nn.Embedding(vocabulary_size, embedding_dim)
        self.word_biases = nn.Embedding(vocabulary_size, 1)
        self.context_biases = nn.Embedding(vocabulary_size, 1)

        nn.init.normal_(self.word_embeddings.weight, std=0.02)
        nn.init.normal_(self.context_embeddings.weight, std=0.02)
        nn.init.zeros_(self.word_biases.weight)
        nn.init.zeros_(self.context_biases.weight)

    def forward(
        self,
        center_ids: torch.Tensor,
        context_ids: torch.Tensor,
        counts: torch.Tensor,
    ) -> torch.Tensor:
        center_vectors = self.word_embeddings(center_ids)
        context_vectors = self.context_embeddings(context_ids)
        center_biases = self.word_biases(center_ids).squeeze(1)
        context_biases = self.context_biases(context_ids).squeeze(1)

        # GLOVE TODO 3: compute predictions, weights, and mean loss.
        # Replace the next line.
        raise NotImplementedError("Return weighted squared error")

<details>
<summary><strong>GloVe Hint 3A — construct the prediction</strong></summary>

The dot product is `(center_vectors * context_vectors).sum(dim=1)`. Add both one-dimensional bias tensors. The result has one prediction per training row.
</details>

<details>
<summary><strong>GloVe Hint 3B — construct the target and loss</strong></summary>

```python
predictions = ...
targets = torch.log(counts)
weights = glove_weight(counts)
return (weights * (predictions - targets).pow(2)).mean()
```

All four tensors should have shape `(batch_size,)`. Do not take `log(0)`: the dataset contains only nonzero counts.
</details>

In [ ]:
test_glove = GloVeModel(vocabulary_size=20, embedding_dim=8)
test_loss = test_glove(
    torch.tensor([1, 2, 3]),
    torch.tensor([4, 5, 6]),
    torch.tensor([1.0, 2.0, 4.0]),
)
assert test_loss.ndim == 0
assert test_loss.requires_grad
assert torch.isfinite(test_loss)
print("✓ GloVe TODO 3 passed")

## Train GloVe

Everything below is supplied. Notice that GloVe receives counts rather than binary labels.

In [ ]:
def train_glove_epoch(model, loader, optimizer) -> float:
    model.train()
    total_loss = 0.0

    for center_ids, context_ids, counts in loader:
        center_ids = center_ids.to(DEVICE)
        context_ids = context_ids.to(DEVICE)
        counts = counts.to(DEVICE)

        optimizer.zero_grad()
        loss = model(center_ids, context_ids, counts)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(counts)

    return total_loss / len(loader.dataset)


GLOVE_EPOCHS = 12
glove_model = GloVeModel(len(vocabulary), EMBEDDING_DIM).to(DEVICE)
glove_optimizer = torch.optim.Adam(glove_model.parameters(), lr=0.03)

glove_history = []
for epoch in range(1, GLOVE_EPOCHS + 1):
    mean_loss = train_glove_epoch(glove_model, glove_loader, glove_optimizer)
    glove_history.append(mean_loss)
    print(f"Epoch {epoch:02d} · loss {mean_loss:.4f}")

assert glove_history[-1] < glove_history[0]

plt.plot(range(1, GLOVE_EPOCHS + 1), glove_history, marker="o", color="#d6a734")
plt.xlabel("Epoch")
plt.ylabel("Mean weighted squared error")
plt.title("GloVe training curve")
plt.show()

## Inspect and visualize GloVe weights

The conventional final vector combines the two learned tables.

In [ ]:
glove_weights = (
    glove_model.word_embeddings.weight + glove_model.context_embeddings.weight
).detach().cpu().numpy()

for query in ["king", "queen", "water", "music"]:
    print(query, nearest_words(glove_weights, query, word_to_id, id_to_word))

glove_plot_matrix = np.stack(
    [glove_weights[word_to_id[word]] for word in learned_words]
)
glove_points = PCA(n_components=2, random_state=SEED).fit_transform(glove_plot_matrix)
plot_embedding(learned_words, glove_points, "GloVe weights learned from global counts")

## Compare all three representations

In [ ]:
comparison_queries = [word for word in ["king", "water", "music"] if word in word_to_id]

rows = []
for query in comparison_queries:
    rows.append(
        {
            "query": query,
            "pretrained_GloVe": [word for word, _ in glove.most_similar(query, topn=5)],
            "our_Word2Vec": [
                word for word, _ in nearest_words(
                    word2vec_weights, query, word_to_id, id_to_word
                )
            ],
            "our_GloVe": [
                word for word, _ in nearest_words(
                    glove_weights, query, word_to_id, id_to_word
                )
            ],
        }
    )

pd.DataFrame(rows)

## Final reflection

Write **300–450 words** addressing all of the following:

1. What evidence does each of the three embedding workflows use?
2. Why is pretrained GloVe substantially better than either model trained here?
3. Compare one Word2Vec neighborhood with the corresponding GloVe neighborhood.
4. What did negative sampling change computationally?
5. Why can a visually convincing PCA plot still be misleading?
6. Identify one limitation shared by all three static embeddings.

## Submission checklist

- [ ] TODOs 1–7 pass their checks.
- [ ] All three GloVe TODOs pass their checks.
- [ ] Training curves and both learned-space plots are visible.
- [ ] The comparison table is visible.
- [ ] The 300–450 word reflection is complete.
- [ ] The notebook has been restarted and run from top to bottom.